# 08 在哪裡？地圖 — 筆記本

面對任何疫情，市長的第一個問題都是「*在哪裡？*」在這本筆記本裡你會：

1. 把登革熱病例數併進 **Santa Luz 市**（虛構；`santa_luz_barangays.geojson`）24 個 barangay（描籠涯，菲律賓最基層的行政區，類似台灣的村里）的多邊形，並計算**侵襲率**；
2. 把病例數與侵襲率並排畫成地圖，看看為什麼它們說出不同的故事；
3. 刻意選擇**分級切點**與**配色**；
4. 看看數字很小的時候，侵襲率有多不穩定；
5. 用 plotly 做一張互動式地圖；
6. 抓出一個悄悄弄丟兩個 barangay 的合併；
7. 拉遠到真實的**菲律賓 18 個行政區**，作為背景；
8. 準備一張可以安全公開的地圖（小格遮蔽）。

關於 Santa Luz 的一切都是虛構的；行政區地圖與它的統計數字則是真實且附有來源的。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## 步驟 1 — 每個 barangay 的病例數與侵襲率

每個 barangay 我們需要兩樣東西：登革熱病例數（來自[第 03 章](../03_clean_linelist.md)乾淨的造冊），以及人口（來自 barangay 檔案）。然後 **每 10,000 人侵襲率 = 病例數 ÷ 人口 × 10,000**。

> **問問 Tala：** *我有一個 GeoDataFrame `brgy`（24 列；欄位有 `barangay_id`、`barangay`、`band`、`population`、`geometry`），還有一份造冊 `ll`（每一列是一位病例；欄位有 `barangay_id`、`disease`）。虛構範例：`ll` 的列有 `("SL01", "dengue")`、`("SL01", "dengue")`、`("SL02", "leptospirosis")`。請依 `barangay_id` 計算登革熱病例數，**用 `barangay_id`** 把計數併進 `brgy`，沒有病例的 barangay 填 0，並新增 `attack_rate` = 病例數 / 人口 × 10,000。結果必須仍然是 24 列。*

In [ ]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -- CJK 字型設定（避免中文標籤顯示成方框 □□□）--
import pathlib
import matplotlib.font_manager as fm

for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# 動態偵測實際註冊的 CJK 字型名稱（避免 .ttc face 0 陷阱）
_discovered = []
for _entry in fm.fontManager.ttflist:
    _nlower = _entry.name.lower()
    if any(_kw in _nlower for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
_cjk_fonts = list(_discovered)
for _n in _preferred:
    if _n not in _cjk_fonts:
        _cjk_fonts.append(_n)
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts
import plotly.express as px
from scipy import stats

brgy = gpd.read_file("data/synthetic/ph/santa_luz_barangays.geojson")
pop = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")[["barangay_id", "households"]]
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

dengue_counts = ll[ll["disease"] == "dengue"].groupby("barangay_id").size().rename("cases")
brgy = brgy.merge(pop, on="barangay_id", how="left")
brgy = brgy.merge(dengue_counts, left_on="barangay_id", right_index=True, how="left")
brgy["cases"] = brgy["cases"].fillna(0).astype(int)        # 沒有病例的 barangay 是 0，不是缺值
brgy["attack_rate"] = brgy["cases"] / brgy["population"] * 10_000

print(f"{len(brgy)} 個 barangay，{brgy['cases'].sum()} 例登革熱")
print(brgy.sort_values("attack_rate", ascending=False)
      [["barangay", "band", "population", "cases", "attack_rate"]].head(5).round(1).to_string(index=False))

**檢查一下。** (1) 合併之後仍是 24 列嗎？(2) 各 barangay 的病例數加起來等於造冊裡全部 388 例登革熱嗎？(3) 手算一個侵襲率：Talaba 有 29 例、6,688 位居民 → 29 ÷ 6,688 × 10,000 = 43.4。(4) 有沒有 **0** 病例的 barangay，它在合併後還在嗎？`groupby` 絕不會為沒有病例的 barangay 建立一列——上面的 `fillna(0)` 才是讓它留下來的原因。

In [ ]:
assert len(brgy) == 24, "合併弄丟或重複了 barangay"
assert brgy["cases"].sum() == (ll["disease"] == "dengue").sum() == 388
talaba = brgy.set_index("barangay").loc["Talaba"]
assert abs(29 / 6688 * 10_000 - talaba["attack_rate"]) < 1e-9
print("Talaba 手算：", round(29 / 6688 * 10_000, 1))
print("登革熱病例為 0 的 barangay：", brgy.loc[brgy["cases"] == 0, "barangay"].tolist())
print("groupby 結果裡的 barangay 數：", len(dengue_counts), "<- 比 24 少一個")

## 步驟 2 — 病例數 vs. 侵襲率

同樣的病例，兩張地圖。左邊是**病例數**：大的 barangay 只因為住的人多就亮起來。右邊是**侵襲率**：居民成為病例的機率。侵襲率回答「*風險在哪裡？*」；病例數回答「*工作量在哪裡？*」兩者都有用——只是絕不要把它們搞混。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, column, title in [(axes[0], "cases", "登革熱病例數"),
                          (axes[1], "attack_rate", "每 10,000 人侵襲率")]:
    brgy.plot(column=column, cmap="YlOrRd", edgecolor="#1c1b29", linewidth=0.6, legend=True,
              legend_kwds={"shrink": 0.6}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
top_count = brgy.loc[brgy["cases"].idxmax()]
top_rate = brgy.loc[brgy["attack_rate"].idxmax()]
for ax, row in [(axes[0], top_count), (axes[1], top_rate)]:
    ax.annotate(row["barangay"], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                ha="center", fontsize=9, fontweight="bold",
                bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "alpha": 0.85})
fig.suptitle("Santa Luz 市（虛構），登革熱，2026 年 7/20 至 9/30", fontsize=12)
plt.tight_layout()
plt.show()
print(f"病例數最多：{top_count['barangay']}（{top_count['band']}，{top_count['cases']} 例，"
      f"每 10,000 人 {top_count['attack_rate']:.1f}）")
print(f"侵襲率最高：{top_rate['barangay']}（{top_rate['band']}，{top_rate['cases']} 例，"
      f"每 10,000 人 {top_rate['attack_rate']:.1f}）")

## 步驟 3 — 分級切點與顏色

面量圖會把數值分成幾個**級別**，而切點會改變訊息：

* **等距分級（equal interval）** — 各級寬度相同（這裡每級約每 10,000 人 11 例）；少數極端的 barangay 可能讓地圖的大部分區域都是同一種顏色；
* **分位數分級（quantile）** — 每一級有相同數量的 barangay（24 ÷ 4 = 6）；每種顏色都用得到，但相鄰的級別可能非常接近。

顏色：請使用**循序式**（sequential）色盤（淺 = 低，深 = 高），例如 `YlOrRd` 或 `viridis`；它們讀起來有順序，色盲也看得懂，灰階列印也不會失真。**彩虹**色盤（`jet`）沒有自然的順序，而且正中間那條亮黃色會把目光拉到*中等*的數值上。

In [ ]:
rate = brgy["attack_rate"]
equal_edges = np.linspace(rate.min(), rate.max(), 5)          # 4 個等寬的級別
quantile_edges = rate.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy()   # 4 個級別，每級 6 個 barangay



def classes(values, edges, digits=1):
    # 把數值切成標示為「低-高」的級別（含最低的邊界）
    labels = [f"{edges[i]:,.{digits}f}-{edges[i + 1]:,.{digits}f}" for i in range(len(edges) - 1)]
    return pd.cut(values, edges, include_lowest=True, labels=labels)


brgy["equal_class"] = classes(rate, equal_edges)
brgy["quantile_class"] = classes(rate, quantile_edges)
print("等距分級：", brgy["equal_class"].value_counts(sort=False).to_dict())
print("分位數分級：", brgy["quantile_class"].value_counts(sort=False).to_dict())

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
panels = [("equal_class", "YlOrRd", "等距分級，循序式色盤"),
          ("quantile_class", "YlOrRd", "分位數分級，循序式色盤"),
          ("quantile_class", "jet", "分位數分級，彩虹色盤（避免）")]
for ax, (column, cmap, title) in zip(axes, panels):
    brgy.plot(column=column, categorical=True, cmap=cmap, edgecolor="#1c1b29", linewidth=0.5,
              legend=True, legend_kwds={"loc": "upper center", "bbox_to_anchor": (0.5, 0.0), "ncol": 2,
                                        "fontsize": 8, "title": "每 10,000 人", "frameon": False}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 步驟 4 — 小數字會亂跳

山區的 barangay 只有 3,000–5,000 位居民，病例也只有寥寥幾例，所以多一例或少一例，侵襲率就會變很多。病例數的**精確 Poisson 95% 信賴區間**可以顯示變動有多大：區間很寬，代表「不要光憑顏色就替這個 barangay 排名」。

In [ ]:
def poisson_ci(count, alpha=0.05):
    # Poisson 計數的精確（Garwood）信賴區間
    low = stats.chi2.ppf(alpha / 2, 2 * count) / 2 if count > 0 else 0.0
    high = stats.chi2.ppf(1 - alpha / 2, 2 * (count + 1)) / 2
    return low, high


ci = brgy["cases"].apply(lambda c: pd.Series(poisson_ci(c), index=["low", "high"]))
brgy["rate_low"] = ci["low"] / brgy["population"] * 10_000
brgy["rate_high"] = ci["high"] / brgy["population"] * 10_000
brgy["one_more_case"] = (brgy["cases"] + 1) / brgy["population"] * 10_000

cols = ["barangay", "band", "population", "cases", "attack_rate", "rate_low", "rate_high", "one_more_case"]
picked = brgy.set_index("barangay").loc[["Talaba", "Ilog", "Maligaya", "Bukid"]].reset_index()
print(picked[cols].round(1).to_string(index=False))

## 步驟 5 — 互動式地圖

在螢幕上（[第 09 章](../09_dashboard.md)的儀表板），互動式地圖讓讀者可以把滑鼠移上去，看確切的數字。Plotly 需要以 GeoJSON 格式提供的多邊形，以及把每一列資料連到對應多邊形的鍵：`featureidkey="properties.barangay_id"`。我們不畫任何底圖：Santa Luz 是虛構的，它的座標刻意落在外海。

In [ ]:
geojson = json.loads(brgy[["barangay_id", "geometry"]].to_json())
fig = px.choropleth(
    brgy.drop(columns="geometry"),
    geojson=geojson,
    locations="barangay_id",
    featureidkey="properties.barangay_id",
    color="attack_rate",
    color_continuous_scale="YlOrRd",
    hover_name="barangay",
    hover_data={"band": True, "cases": True, "population": ":,", "attack_rate": ":.1f", "barangay_id": False},
    labels={"attack_rate": "每 10,000 人", "cases": "病例數", "population": "人口", "band": "區帶"},
    title="每 10,000 人登革熱侵襲率，Santa Luz 市（虛構）",
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"l": 0, "r": 0, "t": 40, "b": 0}, height=420)
fig.show()

## Tala 搞錯了：消失的兩個 barangay

Jun 傳來一份手打的、依 barangay **名稱**列出登革熱病例數的簡單表格。Maya 請 Tala 把它併到地圖上，Tala 就**用名稱**來合併。地圖看起來沒問題——但 Jun 把 `Santo Niño` 打成了 `Santo Nino`（沒有 ñ），把 `Tabing-Ilog` 打成了 `Tabing Ilog`（沒有連字號）。用內部合併（inner join）時，這兩列就這樣消失了：沒有錯誤，也沒有警告。兩個河岸 barangay，合計 41 例，會在市長的地圖上變成一片空白。

把關只要一行：`assert len(merged) == 24`。解法是**用代碼**（`barangay_id`）**來合併**，絕不要用手打的名稱。

In [ ]:
# Jun 手打的表格（同樣的病例數，兩個名稱拼法不同）
jun = brgy[["barangay", "cases"]].copy()
jun["barangay"] = jun["barangay"].replace({"Santo Niño": "Santo Nino", "Tabing-Ilog": "Tabing Ilog"})

merged = brgy[["barangay_id", "barangay", "geometry"]].merge(jun, on="barangay", how="inner")   # Tala 的合併
print("Tala 合併後的列數：", len(merged))
try:
    assert len(merged) == 24, f"join kept {len(merged)} of 24 barangays"
except AssertionError as err:
    print("抓到了：", err)

check = brgy[["barangay"]].merge(jun, on="barangay", how="outer", indicator=True)
print(check[check["_merge"] != "both"].to_string(index=False))

## 步驟 6 — 拉遠：菲律賓的 18 個行政區

在簡報時，把一個省在全國的位置秀出來會有幫助。`ph_regions.topo.json` 存有 18 個行政區真實的邊界，格式是 **TopoJSON**（一種比 GeoJSON 更精簡的表示法）；geopandas 可以直接讀。我們用 PSGC 代碼 `adm1_psgc`（兩個檔案裡都是數字）把 `ph_regions_stats.csv` 裡真實的區域統計數字併進來，並畫出 **2024 年的人口密度**。密度是推算出來的（2024 年人口 ÷ 土地面積），兩個輸入值在 CSV 裡都附有來源網址。NCR 的密度大約是第二名（CALABARZON）的 22 倍，所以如果用等寬分級，其他所有行政區都會是同一種顏色；分位數分級才看得出差異。

這是一張真實國家的**背景**地圖。絕不要把虛構的 Santa Luz 畫在上面。

In [ ]:
regions = gpd.read_file("data/synthetic/ph/ph_regions.topo.json").set_crs("EPSG:4326")
ph_stats = pd.read_csv("data/synthetic/ph/ph_regions_stats.csv", dtype={"region_id": str, "psgc": str})
ph = regions[["adm1_psgc", "adm1_en", "geometry"]].merge(ph_stats, on="adm1_psgc", how="inner")
assert len(ph) == 18, f"合併只留下 18 個行政區中的 {len(ph)} 個"

density = ph["density_2024_per_km2"]
ph["density_class"] = classes(density, density.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy(), digits=0)
print(ph.sort_values("density_2024_per_km2", ascending=False)
      [["region_short", "population_2024", "land_area_km2", "density_2024_per_km2"]].head(4).to_string(index=False))
print("population_2024 的來源：", ph["population_2024_source_url"].iloc[0])

fig, ax = plt.subplots(figsize=(6.5, 8.5))
ph.plot(column="density_class", categorical=True, cmap="YlGnBu", edgecolor="#1c1b29", linewidth=0.4,
        legend=True, legend_kwds={"loc": "upper right", "title": "每平方公里人口，2024 年", "fontsize": 8}, ax=ax)
ax.set_title("菲律賓各行政區人口密度，2024 年")
ax.set_axis_off()
fig.text(0.02, 0.02, "Boundaries: NAMRIA, PSA via OCHA COD-AB (CC BY-IGO); TopoJSON: faeldon/philippines-json-maps (MIT).\n"
         "人口密度 = 2024 年人口 / 土地面積，兩者皆取自 en.wikipedia.org/wiki/Regions_of_the_Philippines。",
         fontsize=6.5)
plt.tight_layout()
plt.show()

## 步驟 7 — 讓它可以安全公開

有病例的住戶**點狀圖（dot map）**，等於是把一份地址清單畫成點：任何認得那條街的人都能指出是哪一棟房子。在 CESU 內部，它可以用來引導 4S 小組；但在公開網頁上，它會讓鄰居被認出來。公開地圖應顯示**彙總的多邊形**（barangay），即使如此，病例是 1–4 的 barangay 仍是一個小格，可能指向某個人。一個常見的規則，也是本書使用的規則：**遮蔽 1–4 的數字**（顯示「<5」），同時也要遮蔽它們的侵襲率，因為侵襲率 × 人口就能把病例數算回來。

In [ ]:
public = brgy[["barangay_id", "barangay", "population", "cases", "attack_rate", "geometry"]].copy()
small = public["cases"].between(1, 4)
public["cases_public"] = public["cases"].astype(str).where(~small, "<5")
public["rate_public"] = public["attack_rate"].round(1).where(~small)        # NaN = 不顯示
print(f"已遮蔽：24 個 barangay 中的 {small.sum()} 個：", public.loc[small, "barangay"].tolist())

fig, ax = plt.subplots(figsize=(7, 4.8))
public.plot(column="rate_public", cmap="YlOrRd", edgecolor="#1c1b29", linewidth=0.5, legend=True,
            legend_kwds={"shrink": 0.6, "label": "每 10,000 人"}, ax=ax,
            missing_kwds={"color": "#eeeeee", "hatch": "///", "edgecolor": "#6e7882", "label": "1-4 例：不顯示"})
ax.set_title("公開版本：每 10,000 人登革熱（1-4 例的區域已遮蔽）")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 換你試試

1. **鉤端螺旋體病。** 對 `disease == "leptospirosis"` 重做步驟 1，並把侵襲率畫成地圖。現在哪個區帶亮起來？*（提示：改一個字串，並記得 `fillna(0)`。）*
2. **五級。** 用 5 個分位數級別重做步驟 3。Talaba 還是獨自在最高級嗎？*（提示：`rate.quantile(np.linspace(0, 1, 6))`。）*
3. **不同的區域變數。** 改畫 `poverty_incidence_2023_pct` 而不是密度，並把 `poverty_incidence_2023_pct_source_url` 欄位裡的來源網址加進版權標示。

In [ ]:
# 1. 鉤端螺旋體病（起手式）
lepto = ll[ll["disease"] == "leptospirosis"].groupby("barangay_id").size().rename("lepto_cases")
brgy_l = brgy[["barangay_id", "barangay", "band", "population", "geometry"]].merge(
    lepto, left_on="barangay_id", right_index=True, how="left")
brgy_l["lepto_cases"] = brgy_l["lepto_cases"].fillna(0).astype(int)
brgy_l["lepto_rate"] = brgy_l["lepto_cases"] / brgy_l["population"] * 10_000
assert len(brgy_l) == 24
by_band = brgy_l.groupby("band")[["lepto_cases", "population"]].sum()   # 先加總：侵襲率的平均值不是該區帶的侵襲率
print((by_band["lepto_cases"] / by_band["population"] * 10_000).round(1))

## 你完成了什麼

* **用代碼**把病例數併進多邊形，保留零病例的 barangay，並 assert 剛好 24 列。
* 問的是風險時，畫侵襲率而不是病例數；刻意選擇分級切點與循序式色盤。
* 把很寬的信賴區間讀成「不要光憑顏色排名」。
* 畫出真實的 18 個行政區背景地圖，並附上版權標示。
* 準備一張遮蔽了小格的公開地圖。

下一步：[第 09 章](../09_dashboard.md)把曲線、閾值與這張地圖放在同一個畫面上。